**Partition Data for `n-shot` Eval**

The cell below takes an `IN_DIR` pointing to the *mcd_dataset* folder as input. The cell will create 3 csv files and write them to `OUT_DIR`:

1. *4-shot.csv*: consists of a random selection of 2 MCD-positive and 2 MCD-negative entries from `IN_DIR/metadata.csv` (MCD-positive entries have a *label* of $1$, others a $0$)
2. *8-shot.csv*: consists of a random selection of 2 additional MCD-positive and MCD-negative entries from `IN_DIR/metadata.csv` (MCD-positive entries have a *label* of $1$, others a $0$; *4-shot.csv* is a subset of *8-shot.csv*)
2. *eval.csv*: consists of a random selection of 30 entries from `IN_DIR/metadata.csv` not in *8-shot.csv* (MCD-positive entries have a *label* of $1$, others a $0$)

In [ ]:
IN_DIR = "/scratch/gpfs/FELLBAUM/af3158/cos351/mcd_dataset"
OUT_DIR = "/scratch/gpfs/FELLBAUM/af3158/cos351/closed_weight_n_shot/partitioned_data"

In [ ]:
import os
import pandas as pd

df = pd.read_csv(os.path.join(IN_DIR, "metadata.csv"))

positives = df[df["label"] == 1]
negatives = df[df["label"] == 0]

# 4-shot: 2 random positive + 2 random negative
four_pos = positives.sample(n=2, random_state=42)
four_neg = negatives.sample(n=2, random_state=42)
four_shot = pd.concat([four_pos, four_neg])

# 8-shot: 4-shot + 2 additional random positive + 2 additional random negative
remaining_pos = positives.drop(four_pos.index)
remaining_neg = negatives.drop(four_neg.index)
extra_pos = remaining_pos.sample(n=2, random_state=42)
extra_neg = remaining_neg.sample(n=2, random_state=42)
eight_shot = pd.concat([four_shot, extra_pos, extra_neg])

# eval: 30 random entries not in 8-shot
remaining = df.drop(eight_shot.index)
eval_set = remaining.sample(n=30, random_state=42)

# write to OUT_DIR
os.makedirs(OUT_DIR, exist_ok=True)
four_shot.to_csv(os.path.join(OUT_DIR, "4-shot.csv"), index=False)
eight_shot.to_csv(os.path.join(OUT_DIR, "8-shot.csv"), index=False)
eval_set.to_csv(os.path.join(OUT_DIR, "eval.csv"), index=False)

print(f"4-shot: {len(four_shot)} entries ({four_shot['label'].sum()} pos, {(four_shot['label'] == 0).sum()} neg)")
print(f"8-shot: {len(eight_shot)} entries ({eight_shot['label'].sum()} pos, {(eight_shot['label'] == 0).sum()} neg)")
print(f"eval:   {len(eval_set)} entries ({eval_set['label'].sum()} pos, {(eval_set['label'] == 0).sum()} neg)")